# LLM Cost Killer - MVP Showcase

This notebook demonstrates:

- Rule-based smart routing (cheap vs strong model)
- Fallback behavior
- Per-request token and cost tracking
- Naive vs optimized cost comparison

> This demo uses the built-in `MockLLMProvider`, so it runs offline with no API key.

In [1]:
from pathlib import Path
import sys

# Ensure imports work when running notebook from /examples
repo_root = Path.cwd().resolve().parent if Path.cwd().name == "examples" else Path.cwd().resolve()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from lck.router import CostOptimizedRouter, MockLLMProvider
from lck.cost_tracker import CostTracker, estimate_request_cost
from lck.config import DEFAULT_STRONG_MODEL

print("Imports ready.")

Imports ready.


In [2]:
prompts = [
    "classify this support ticket",
    "summarize this invoice",
    "extract payment terms",
    "compare risks in this contract",
    "write Python code to parse a CSV",
]

provider = MockLLMProvider()
tracker = CostTracker(log_path="logs/notebook_requests.jsonl")
router = CostOptimizedRouter(provider=provider, tracker=tracker)

prompts

['classify this support ticket',
 'summarize this invoice',
 'extract payment terms',
 'compare risks in this contract',
 'write Python code to parse a CSV']

## 1) See routing decisions

This cell shows which model gets selected before any fallback.

In [3]:
for p in prompts:
    chosen = router.choose_model(p)
    print(f"{p!r} -> {chosen}")

'classify this support ticket' -> gpt-4.1-mini
'summarize this invoice' -> gpt-4.1-mini
'extract payment terms' -> gpt-4.1-mini
'compare risks in this contract' -> gpt-4.1
'write Python code to parse a CSV' -> gpt-4.1


## 2) Run optimized mode (router + fallback + logging)

In [4]:
optimized_results = []
for p in prompts:
    out = router.run(p)
    optimized_results.append(out)
    print(
        f"prompt={p!r}\n"
        f"  selected={out['selected_model']} final={out['final_model']} fallback={out['fallback_used']}\n"
        f"  tokens(in/out)=({out['input_tokens']}/{out['output_tokens']}) cost=${out['estimated_cost']:.6f}\n"
    )

session = tracker.session_summary()
print("Session summary:", session)

prompt='classify this support ticket'
  selected=gpt-4.1-mini final=gpt-4.1-mini fallback=False
  tokens(in/out)=(8/14) cost=$0.000051

prompt='summarize this invoice'
  selected=gpt-4.1-mini final=gpt-4.1-mini fallback=False
  tokens(in/out)=(8/14) cost=$0.000051

prompt='extract payment terms'
  selected=gpt-4.1-mini final=gpt-4.1-mini fallback=False
  tokens(in/out)=(8/14) cost=$0.000051

prompt='compare risks in this contract'
  selected=gpt-4.1 final=gpt-4.1 fallback=False
  tokens(in/out)=(8/40) cost=$0.000640

prompt='write Python code to parse a CSV'
  selected=gpt-4.1 final=gpt-4.1 fallback=False
  tokens(in/out)=(9/40) cost=$0.000645

Session summary: {'requests': 5, 'total_input_tokens': 41, 'total_output_tokens': 122, 'total_estimated_cost': 0.0014386, 'fallback_count': 0, 'requests_by_model': {'gpt-4.1-mini': 3, 'gpt-4.1': 2}}


## 3) Compare naive vs optimized cost

In [5]:
naive_total = 0.0
optimized_total = 0.0

for p in prompts:
    naive = provider.generate(prompt=p, model=DEFAULT_STRONG_MODEL)
    naive_total += estimate_request_cost(
        DEFAULT_STRONG_MODEL,
        int(naive["input_tokens"]),
        int(naive["output_tokens"]),
    )

for r in optimized_results:
    optimized_total += float(r["estimated_cost"])

savings = max(0.0, naive_total - optimized_total)
savings_pct = (savings / naive_total * 100.0) if naive_total else 0.0

print(f"naive strong-only: ${naive_total:.6f}")
print(f"optimized routed:  ${optimized_total:.6f}")
print(f"savings:           ${savings:.6f} ({savings_pct:.2f}%)")

naive strong-only: $0.003205
optimized routed:  $0.001439
savings:           $0.001766 (55.11%)


## 4) Inspect the JSONL request log

In [6]:
log_file = repo_root / "logs" / "notebook_requests.jsonl"
print(f"Log file: {log_file}")

if log_file.exists():
    lines = log_file.read_text(encoding="utf-8").strip().splitlines()
    print(f"Total logged requests: {len(lines)}")
    print("Last 3 entries:")
    for line in lines[-3:]:
        print(line)
else:
    print("No log file yet. Run the optimized cell first.")

Log file: /Users/gershonc/Desktop/Workspace/llm-cost-killer/logs/notebook_requests.jsonl
No log file yet. Run the optimized cell first.
